# Phase 4 modern sentiment benchmark — free Colab runner
This notebook runs a bounded number of genuine final experiments. Lightweight evidence is pushed to GitHub and selected checkpoints to a private Hugging Face repository. Select a free GPU runtime before executing.

In [ ]:
import os, pathlib, subprocess
from google.colab import userdata
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['PHASE4_HF_REPO'] = userdata.get('PHASE4_HF_REPO')
github_repo = userdata.get('PHASE4_GITHUB_REPO')
git_name = userdata.get('PHASE4_GIT_NAME')
git_email = userdata.get('PHASE4_GIT_EMAIL')
assert github_repo and git_name and git_email, 'Configure PHASE4_GITHUB_REPO, PHASE4_GIT_NAME, and PHASE4_GIT_EMAIL in Colab secrets.'
PROJECT = '/content/sentimental-analysis-modern'
subprocess.run(['gh', 'auth', 'setup-git'], check=True)
if not pathlib.Path(PROJECT).exists():
    subprocess.run(['git', 'clone', f'https://github.com/{github_repo}.git', PROJECT], check=True)
subprocess.run(['git', 'config', 'user.name', git_name], cwd=PROJECT, check=True)
subprocess.run(['git', 'config', 'user.email', git_email], cwd=PROJECT, check=True)

In [ ]:
# results/modern is lightweight and is persisted by Git; checkpoints go to PHASE4_HF_REPO.
RESULTS_ROOT = f'{PROJECT}/results/modern'
%cd $PROJECT

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'BLOCKED — GPU UNAVAILABLE'
print(torch.cuda.get_device_name(0))

In [ ]:
# Install only the locked stack. Restart the runtime if Colab requests it.
!python -m pip install -r requirements-transformer-smoke.txt

In [ ]:
# Verify only presence; never print secret values.
assert os.environ.get('HF_TOKEN') and os.environ.get('PHASE4_HF_REPO')

In [ ]:
# Run one experiment; rerun this cell/session to continue the queue.
!python scripts/run_phase4_free_gpu.py --results-root "$RESULTS_ROOT" --max-runs 1 --sync-github

In [ ]:
!python -m ml.modern_benchmark audit --results-root "$RESULTS_ROOT"

In [ ]:
# This succeeds only after all 24 validated runs are complete.
!python -m ml.modern_benchmark analyze --results-root "$RESULTS_ROOT"